In [1]:
pip install fastf1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires websockets>=14.0, but you have websockets 13.1 which is incompatible.
google-adk 1.17.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 13.1 which is incompatible.


In [2]:
import fastf1 as ff1
import pandas as pd
import numpy as np
from fastf1 import plotting
from sklearn.preprocessing import StandardScaler

# Configura el plotting de FastF1
plotting.setup_mpl()

# --- 1. Carga de Datos ---
import os
if not os.path.exists('cache'):
    os.makedirs('cache')
ff1.Cache.enable_cache('cache')
session = ff1.get_session(2025, 'Italian Grand Prix', 'R')
session.load()
print(f"Sesión cargada: {session.event.EventName} {session.name}")

core           INFO 	Loading data for Italian Grand Prix - Race [v3.6.1]
INFO:fastf1.fastf1.core:Loading data for Italian Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_d

Sesión cargada: Italian Grand Prix Race


In [3]:
# --- 2. Definición de Zonas de Interés (ZOI) ---
# Definimos zonas de análisis con sub-partes: 'braking', 'apex', 'exit'
circuit_info = session.get_circuit_info()
analysis_zones = {} # Initialize analysis_zones as an empty dictionary
print(f"Total corners found: {len(circuit_info.corners)}")

# Safely define analysis zones by checking if corner indices exist
try:
    if len(circuit_info.corners) > 1: # Ensure there are at least 2 corners for T1/T4 analysis
        analysis_zones['T1_Rettifilo'] = {
            'braking': (circuit_info.corners.loc[0]['Distance'] - 350, circuit_info.corners.loc[0]['Distance']),
            'apex': (circuit_info.corners.loc[0]['Distance'] - 50, circuit_info.corners.loc[1]['Distance'] + 50),
            'exit': (circuit_info.corners.loc[1]['Distance'], circuit_info.corners.loc[1]['Distance'] + 250)
        }
    else:
        print("Not enough corners found to define T1_Rettifilo analysis zone.")

    if len(circuit_info.corners) > 4: # Ensure there are at least 5 corners for T4 analysis
        analysis_zones['T4_Roggia'] = {
            'braking': (circuit_info.corners.loc[3]['Distance'] - 300, circuit_info.corners.loc[3]['Distance']),
            'apex': (circuit_info.corners.loc[3]['Distance'] - 50, circuit_info.corners.loc[4]['Distance'] + 50),
            'exit': (circuit_info.corners.loc[4]['Distance'], circuit_info.corners.loc[4]['Distance'] + 250)
        }
    else:
        print("Not enough corners found to define T4_Roggia analysis zone.")

    if len(circuit_info.corners) > 10: # Ensure there are at least 11 corners for T11 analysis
         analysis_zones['T11_Alboreto'] = {
            'braking': (circuit_info.corners.loc[10]['Distance'] - 350, circuit_info.corners.loc[10]['Distance']),
            'apex': (circuit_info.corners.loc[10]['Distance'] - 100, circuit_info.corners.loc[10]['Distance'] + 100),
            'exit': (circuit_info.corners.loc[10]['Distance'] + 100, circuit_info.corners.loc[10]['Distance'] + 350)
        }
    else:
        print("Not enough corners found to define T11_Alboreto analysis zone.")

except Exception as e:
    print(f"An error occurred while defining analysis zones: {e}")


print(f"Defined {len(analysis_zones)} analysis zones.")

Total corners found: 11
Defined 3 analysis zones.


In [4]:
# --- 3. Funciones de Feature Engineering ---

def get_braking_point(telemetry_fragment):
    """Devuelve la distancia donde 'Brake' se activa."""
    try:
        # Ensure the fragment is not empty and has the 'Brake' column
        if telemetry_fragment.empty or 'Brake' not in telemetry_fragment.columns:
            return np.nan
        braking_start_row = telemetry_fragment[telemetry_fragment['Brake'] > 0.1].first_valid_index()
        if braking_start_row is not None:
            return telemetry_fragment.loc[braking_start_row, 'Distance']
    except Exception as e:
        pass
    return np.nan

def get_min_apex_speed(telemetry_fragment):
    """Devuelve la velocidad mínima en el fragmento de vértice."""
    try:
        if not telemetry_fragment.empty and 'Speed' in telemetry_fragment.columns:
            return telemetry_fragment['Speed'].min()
    except Exception as e:
        pass
    return np.nan

def get_max_deceleration(telemetry_fragment):
    """Devuelve la desaceleración máxima (en m/s^2) mientras se frena."""
    try:
        # Ensure the fragment is not empty and has 'Brake' and 'Acceleration' columns
        if telemetry_fragment.empty or 'Brake' not in telemetry_fragment.columns or 'Acceleration' not in telemetry_fragment.columns:
            return np.nan
        # Filtra solo los momentos en que el freno está aplicado
        braking_fragment = telemetry_fragment[telemetry_fragment['Brake'] > 0.1]
        if not braking_fragment.empty:
            # Encuentra el valor mínimo de aceleración (máxima desaceleración)
            return braking_fragment['Acceleration'].min()
    except Exception as e:
        pass
    return np.nan

def get_max_throttle_rate(telemetry_fragment):
    """Devuelve la tasa máxima de aplicación del acelerador (% por segundo)."""
    try:
        if not telemetry_fragment.empty and 'Throttle' in telemetry_fragment.columns and 'TimeDelta' in telemetry_fragment.columns:
            # Tasa de cambio del acelerador (0-100) por segundo
            # Avoid division by zero if TimeDelta is 0
            throttle_rate = telemetry_fragment['Throttle'].diff() / telemetry_fragment['TimeDelta'].replace(0, np.nan)

            # Encuentra la tasa positiva máxima
            max_rate = throttle_rate[throttle_rate > 0].max()

            if pd.notna(max_rate) and not np.isinf(max_rate):
                return max_rate
    except Exception as e:
        pass
    return np.nan


In [5]:
# --- 4. Iteración por todos los pilotos y promedio de vueltas ---

driver_list = session.drivers
print(f"Analizando {len(driver_list)} pilotos...")

all_driver_features_avg = [] # Lista para guardar el PROMEDIO de features de cada piloto

for driver_code in driver_list:
    # 1. Obtenemos TODAS las vueltas del piloto (quitamos el filtro pick_quicklaps())
    all_laps = session.laps.pick_drivers(driver_code)

    if all_laps.empty:
        print(f"Saltando a {driver_code} (sin vueltas válidas para analizar)")
        continue

    lap_features_list = [] # Lista temporal para las features de CADA vuelta de este piloto

    # 2. Iterar por cada vuelta de ESE piloto
    for lap_index, lap in all_laps.iterlaps():

        try:
            telemetry = lap.get_car_data().add_distance()

            # --- Pre-cálculo de Aceleración y TimeDelta (¡IMPORTANTE!) ---
            telemetry['Speed_ms'] = telemetry['Speed'] / 3.6  # Convertir km/h a m/s
            telemetry['TimeDelta'] = telemetry['Time'].diff().dt.total_seconds()
            telemetry['SpeedDelta'] = telemetry['Speed_ms'].diff()
            # Aceleración en m/s^2
            telemetry['Acceleration'] = telemetry['SpeedDelta'] / telemetry['TimeDelta']

            if telemetry.empty:
                # print(f"Saltando vuelta {lap.LapNumber} de {driver_code} (telemetría vacía)")
                continue

            lap_data = {} # Usamos un dict para esta vuelta

            for zone_name, zone_parts in analysis_zones.items():

                # --- 1. Fragmentar la telemetría según las sub-zonas ---
                braking_fragment = telemetry.loc[
                    (telemetry['Distance'] >= zone_parts['braking'][0]) & (telemetry['Distance'] <= zone_parts['braking'][1])
                ]
                apex_fragment = telemetry.loc[
                    (telemetry['Distance'] >= zone_parts['apex'][0]) & (telemetry['Distance'] <= zone_parts['apex'][1])
                ]
                exit_fragment = telemetry.loc[
                    (telemetry['Distance'] >= zone_parts['exit'][0]) & (telemetry['Distance'] <= zone_parts['exit'][1])
                ]

                # --- 2. Calcular y guardar cada feature de la vuelta ---
                lap_data[f'{zone_name}_BrakingPoint'] = get_braking_point(braking_fragment)
                lap_data[f'{zone_name}_MaxDecel'] = get_max_deceleration(braking_fragment)
                lap_data[f'{zone_name}_ApexSpeed'] = get_min_apex_speed(apex_fragment)
                lap_data[f'{zone_name}_ThrottleRate'] = get_max_throttle_rate(exit_fragment)

            lap_features_list.append(lap_data)

        except Exception as e:
            # print(f"Error procesando vuelta {lap.LapNumber} de {driver_code}: {e}")
            continue


    # --- 3. Calcular el promedio para el piloto ---
    if not lap_features_list: # Si no se pudo calcular features para ninguna vuelta
        print(f"Saltando a {driver_code} (no se pudieron calcular features para ninguna vuelta)")
        continue

    # Convertimos la lista de features de vueltas en un DataFrame
    df_laps_driver = pd.DataFrame(lap_features_list)

    # Calculamos la media de todas las features en todas las vueltas, excluyendo NaNs
    driver_avg_features = df_laps_driver.mean(skipna=True) # Esto nos da una Serie con el promedio

    # Añadimos el código del piloto
    driver_avg_features['Driver'] = driver_code

    # Añadimos esta fila de promedios a nuestra lista maestra
    all_driver_features_avg.append(driver_avg_features)

Analizando 20 pilotos...


In [9]:

# --- 5. Creación y Normalización del DataFrame ---

# Convertimos la lista de Series promediadas en un DataFrame
# Usamos dropna(how='all') para quitar pilotos que solo tienen NaNs
df_raw_features = pd.DataFrame(all_driver_features_avg).set_index('Driver').dropna(how='all')

print("\n--- DataFrame de Features (Datos Promediados) ---")
print(df_raw_features)

if df_raw_features.empty:
    print("No hay datos suficientes para normalizar. Revisa la carga de datos.")
else:
    # Calculamos la media de cada feature (columna)
    feature_averages = df_raw_features.mean()

    # NORMALIZACIÓN: Restamos la media de la feature a cada piloto
    df_normalized = df_raw_features - feature_averages

    print("\n--- DataFrame Normalizado (Puntuación de Estilo Promedio) ---")
    print(df_normalized)
    print("\nInterpretación del DataFrame Normalizado:")
    print("Valores positivos = 'más que el promedio' (ej. frenó más tarde, fue más rápido en vértice, etc.)")
    print("Valores negativos = 'menos que el promedio'")




--- DataFrame de Features (Datos Promediados) ---
        T1_Rettifilo_BrakingPoint  T1_Rettifilo_MaxDecel  \
Driver                                                     
1                      768.633333             -45.643508   
4                      776.431661             -43.505506   
81                     765.986965             -40.183507   
16                     775.733705             -48.495516   
63                     767.617835             -45.295027   
44                     763.853160             -45.812347   
23                     773.920357             -47.192878   
5                      767.410068             -42.253800   
12                     769.003763             -44.898436   
6                      751.191971             -42.886789   
55                     749.506536             -44.956164   
87                     749.587070             -42.516161   
22                     760.583464             -40.543232   
30                     761.338061             -40

In [7]:
    # --- 6. ESTANDARIZACIÓN (Paso para Clustering) ---

    print("\n--- Estandarizando datos para Clustering ---")

    # Eliminamos columnas con NaN o Inf que puedan causar problemas en StandardScaler
    df_normalized_cleaned = df_normalized.replace([np.inf, -np.inf], np.nan).dropna(axis=1)

    if df_normalized_cleaned.empty:
        print("No hay columnas válidas para estandarizar después de limpiar NaNs/Infs.")
    else:
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_normalized_cleaned)

        # Creamos un nuevo DataFrame con los datos estandarizados
        df_standardized = pd.DataFrame(
            data=scaled_data,
            index=df_normalized_cleaned.index,
            columns=df_normalized_cleaned.columns
        )

        print("\n--- DataFrame Estandarizado (Listo para Clustering/PCA/t-SNE) ---")
        print(df_standardized) # Mostrar head para evitar salida excesiva
        print("\nInterpretación del DataFrame Estandarizado:")
        print("Todas las features están ahora en la misma escala (Z-scores, media=0, std=1).")
        print("Este es el input final para tu análisis de machine learning.")
        print("Este es el input final para tu análisis de machine learning.")


--- Estandarizando datos para Clustering ---

--- DataFrame Estandarizado (Listo para Clustering/PCA/t-SNE) ---
        T1_Rettifilo_BrakingPoint  T1_Rettifilo_MaxDecel  \
Driver                                                     
1                        0.634975              -0.926666   
4                        1.401424              -0.100084   
81                       0.374880               1.184248   
16                       1.332826              -2.029293   
63                       0.535168              -0.791938   
44                       0.165161              -0.991942   
23                       1.154604              -1.525675   
5                        0.514748               0.383844   
12                       0.671382              -0.638610   
6                       -1.079228               0.139121   
55                      -1.244879              -0.660929   
87                      -1.236964               0.282411   
22                      -0.156197              